# Entrenamiento baseline en Colab

Este notebook solo orquesta: la lógica está en el paquete `defect_detection` y se lanza con `scripts/train.py`, igual que en local.

Antes de ejecutar:

1. **Entorno con GPU**: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.
2. **Dataset en Drive**: sube `data/processed/hazelnut.zip` (generado con `scripts/package_dataset.py pack`) a `Mi unidad/industrial-defect-detection/`.

## 1. GPU disponible

In [ ]:
!nvidia-smi

## 2. Código

Se trabaja siempre en `/content`, de modo que la celda se puede repetir: si el repo ya está clonado se actualiza en vez de clonarse otra vez. Se anota el commit para saber con qué versión del código se entrenó.

In [ ]:
REPO = "https://github.com/olijuseju/industrial-defect-detection.git"
BRANCH = "main"

%cd /content
!test -d industrial-defect-detection/.git || git clone --branch {BRANCH} {REPO}
%cd /content/industrial-defect-detection
!git pull --ff-only origin {BRANCH}
!git rev-parse HEAD

## 3. Dependencias

`pip` ignora `[tool.uv.sources]`, así que se usa el torch con CUDA de Colab en vez de la build CPU que se fija en local.

In [ ]:
!python --version
%pip install -q -e .
!python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())"

## 4. Dataset

Se verifica el SHA256 del zip contra `configs/dataset_checksums.json` antes de extraerlo: si no coincide, no es el dataset con el que se generó el split.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/industrial-defect-detection"

!python scripts/package_dataset.py unpack "{DRIVE_DIR}/hazelnut.zip"

## 5. Entrenamiento

`CONFIG` elige el experimento; los parámetros están en ese fichero. El run se guarda en `runs/<name>`, con el `name` de la config. Si ya existe uno con ese nombre en esta máquina se borra, porque Ultralytics lo guardaría con otro nombre (`<name>2`).

In [ ]:
import yaml

CONFIG = "configs/train_baseline.yaml"
RUN = yaml.safe_load(open(CONFIG))["name"]

!rm -rf "runs/{RUN}"
!python scripts/train.py --config {CONFIG} --device 0

## 6. Guardar el run en Drive

La máquina de Colab se borra al cerrar la sesión. Se copia el run completo (pesos, `results.csv` y gráficas), con el commit en el nombre.

In [ ]:
!mkdir -p "{DRIVE_DIR}/runs"
!cp -r "runs/{RUN}" "{DRIVE_DIR}/runs/{RUN}-$(git rev-parse --short HEAD)"
!ls "{DRIVE_DIR}/runs"